# apatch — Рефакторинг для разработчиков

- Вырезание блоков по маркерам (`apatch strip`)
- Фиксация правок в TrustChain

> Всё реально. Никаких моков.

In [ ]:
import os, sys, json, shutil, tempfile, hashlib

SANDBOX = os.path.join(tempfile.gettempdir(), 'apatch_human_sandbox')
if os.path.exists(SANDBOX):
    shutil.rmtree(SANDBOX)
os.makedirs(SANDBOX)
print(f'Sandbox: {SANDBOX}')

In [ ]:
api_path = os.path.join(SANDBOX, 'api_service.py')
with open(api_path, 'w') as f:
    f.write(
        '# Large Legacy API Service\n'
        '\n'
        'class LegacyAPIService:\n'
        '    def __init__(self, port: int):\n'
        '        self.port = port\n'
        '        self.status = "stopped"\n'
        '\n'
        '    def handle_request(self, req):\n'
        '        if req["path"] == "/health":\n'
        '            return {"status": "ok"}\n'
        '        return {"error": "not_found"}\n'
        '\n'
        '    # BEGIN EXPERIMENTAL HELPER\n'
        '    def run_experimental_eval(self, data):\n'
        '        print("Running heavy experimental metrics...")\n'
        '        return list(map(lambda x: x * 42, data))\n'
        '    # END EXPERIMENTAL HELPER\n'
    )
print('Legacy файл:')
print(open(api_path).read())

## Вырезание блока по маркерам

In [ ]:
from apatch.strip import StripSpec, apply_strips

spec = StripSpec(
    label='experimental_helper',
    start='# BEGIN EXPERIMENTAL HELPER',
    until='# END EXPERIMENTAL HELPER',
    replace='    # [STUBBED] moved to eval_service.py\n'
)

new_lines, results = apply_strips(api_path, [spec])

# Записываем результат
with open(api_path, 'w') as f:
    f.writelines(new_lines)

print('После strip:')
print(open(api_path).read())

content = open(api_path).read()
assert 'run_experimental_eval' not in content
assert 'STUBBED' in content
print('Экспериментальный код убран, заглушка на месте!')

## Фиксация в TrustChain

In [ ]:
from trustchain import TrustChain, TrustChainConfig

tc_dir = os.path.join(SANDBOX, '.trustchain')
cfg = TrustChainConfig(enable_chain=True, chain_storage='file', chain_dir=tc_dir)
tc = TrustChain(cfg)

with open(api_path, 'rb') as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

receipt = tc.sign(tool_id='developer_refactor', data={
    'action': 'strip_experimental_block',
    'file': 'api_service.py',
    'sha256': sha256
})
print(f'TrustChain OK, sha256={sha256[:32]}...')

In [ ]:
shutil.rmtree(SANDBOX)
print('Sandbox удалён.')